In [1]:
from google.colab import files

uploaded = files.upload()

Saving lora_adapter.zip to lora_adapter.zip


In [2]:
from google.colab import files

uploaded = files.upload()

Saving dev_final.json to dev_final.json


In [4]:
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3

import json, zipfile, os, gc, time
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from peft import PeftModel

# ── 1. Unzip planner adapter ──────────────────────────────────
ADAPTER_ZIP = "./lora_adapter.zip"       # <-- upload this (29 MB)
ADAPTER_DIR = "./lora_adapter_planner"
DEV_PATH    = "./dev_final.json"         # <-- upload this
OUTPUT_PATH = "./dev_500_with_pred_plans.json"
N_SAMPLES   = 150
MAX_INPUT   = 512
MAX_TARGET  = 384

if not os.path.exists(ADAPTER_DIR):
    with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
        z.extractall(ADAPTER_DIR)
    print("Extracted to:", ADAPTER_DIR)

# ── 2. Load base model + planner LoRA ────────────────────────
MODEL_NAME = "google/flan-t5-xl"

# Use float16 for T4 (bfloat16 is A100 only)
dtype = torch.bfloat16 if torch.cuda.is_available() and \
        torch.cuda.get_device_capability()[0] >= 8 else torch.float16

print(f"Loading base model ({dtype})...")
base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype = dtype,
    device_map  = "auto",
)

print("Loading planner LoRA...")
planner = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
planner.eval()
planner.config.use_cache = True

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print("Planner ready!")

# ── 3. Load dev data ──────────────────────────────────────────
with open(DEV_PATH, encoding="utf-8") as f:
    dev_data = json.load(f)

dev_500 = dev_data[:N_SAMPLES]
print(f"Loaded {len(dev_500)} dev samples")

# ── 4. Generate plans ─────────────────────────────────────────
def generate_plan(input_text):
    inp = tokenizer(
        input_text,
        return_tensors = "pt",
        max_length     = MAX_INPUT,
        truncation     = True,
    )
    inp = {k: v.to(planner.device) for k, v in inp.items()}
    with torch.no_grad():
        out = planner.generate(
            **inp,
            max_new_tokens = MAX_TARGET,
            num_beams      = 4,
            early_stopping = True,
            length_penalty = 1.0,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

results = []
start = time.time()

for i, sample in enumerate(dev_500):
    pred_plan = generate_plan(sample["input"])
    results.append({
        "idx"       : i,
        "input"     : sample["input"],
        "gold_plan" : sample["output"],   # ground truth plan
        "pred_plan" : pred_plan,          # predicted plan  ← fed to plan2sql next
        "gold_sql"  : sample["sql"],      # ground truth SQL (for final F1)
        "db_id"     : sample.get("db_id", ""),
    })

    # progress every 50
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start
        eta = elapsed / (i + 1) * (N_SAMPLES - i - 1)
        print(f"  [{i+1}/{N_SAMPLES}]  elapsed: {elapsed/60:.1f} min  ETA: {eta/60:.1f} min")

# ── 5. Save ───────────────────────────────────────────────────
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nDone! Saved {len(results)} samples → {OUTPUT_PATH}")
print(f"Total time: {(time.time()-start)/60:.1f} min")

# Quick sanity check
print("\n── Sample 0 ──")
print("INPUT     :", results[0]["input"][:80], "...")
print("GOLD PLAN :", results[0]["gold_plan"][:80], "...")
print("PRED PLAN :", results[0]["pred_plan"][:80], "...")
print("GOLD SQL  :", results[0]["gold_sql"])

Extracted to: ./lora_adapter_planner
Loading base model (torch.float16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading planner LoRA...
Planner ready!
Loaded 150 dev samples
  [50/150]  elapsed: 6.1 min  ETA: 12.3 min
  [100/150]  elapsed: 14.1 min  ETA: 7.0 min
  [150/150]  elapsed: 21.2 min  ETA: 0.0 min

Done! Saved 150 samples → ./dev_500_with_pred_plans.json
Total time: 21.2 min

── Sample 0 ──
INPUT     : question: How many singers do we have? | schema: singer ( Singer_ID [PK] ) | for ...
GOLD PLAN : step1: SCAN | table: singer || step2: AGGREGATE | Scalar aggregate (no GROUP BY) ...
PRED PLAN : step1: SCAN | table: singer || step2: AGGREGATE | Scalar aggregate (no GROUP BY) ...
GOLD SQL  : SELECT count(*) FROM singer


In [5]:
def tokenize_plan(text):
    import re
    return re.findall(r'\w+', text.lower())

def token_f1(pred, gold):
    pred_tokens = tokenize_plan(pred)
    gold_tokens = tokenize_plan(gold)
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0
    from collections import Counter
    pred_count = Counter(pred_tokens)
    gold_count = Counter(gold_tokens)
    overlap    = sum((pred_count & gold_count).values())
    precision  = overlap / len(pred_tokens)
    recall     = overlap / len(gold_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# ── Compute scores ────────────────────────────────────────────
f1_scores = []

for r in results:
    f1 = token_f1(r["pred_plan"], r["gold_plan"])
    f1_scores.append(f1)
    r["plan_f1"] = round(f1, 4)

avg_f1 = sum(f1_scores) / len(f1_scores)

print("══════════════════════════════════════")
print("       PLANNER EVALUATION (n=150)     ")
print("══════════════════════════════════════")
print(f"  Token F1 : {avg_f1*100:.2f}%")
print("══════════════════════════════════════")

# ── Save updated results ──────────────────────────────────────
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Scores saved to {OUTPUT_PATH}")

# ── Show a few examples ───────────────────────────────────────
print("\n── Sample predictions ──")
for i in [0, 10, 50]:
    r = results[i]
    print(f"\n[{i}] F1={r['plan_f1']:.3f}")
    print(f"  GOLD: {r['gold_plan']}")
    print(f"  PRED: {r['pred_plan']}")

══════════════════════════════════════
       PLANNER EVALUATION (n=150)     
══════════════════════════════════════
  Token F1 : 95.18%
══════════════════════════════════════
Scores saved to ./dev_500_with_pred_plans.json

── Sample predictions ──

[0] F1=1.000
  GOLD: step1: SCAN | table: singer || step2: AGGREGATE | Scalar aggregate (no GROUP BY)  ->  compute COUNT(*) || step3: PROJECT | columns: COUNT(*)
  PRED: step1: SCAN | table: singer || step2: AGGREGATE | Scalar aggregate (no GROUP BY) -> compute COUNT(*) || step3: PROJECT | columns: COUNT(*)

[10] F1=1.000
  GOLD: step1: SCAN | table: singer || step2: AGGREGATE | group_by: country | compute: COUNT(*) || step3: PROJECT | columns: country, COUNT(*)
  PRED: step1: SCAN | table: singer || step2: AGGREGATE | group_by: Country | compute: COUNT(*) || step3: PROJECT | columns: Country, COUNT(*)

[50] F1=1.000
  GOLD: step1: SCAN | table: pets || step2: AGGREGATE | group_by: petType | compute: MAX(weight) || step3: PROJECT | columns:

In [8]:
from google.colab import files
files.download("dev_500_with_pred_plans.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>